### ECCV 2014 ~2018

# 2018

In [3]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [4]:
import re
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 ICDM 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 섹션 찾기
    for entry in soup.find_all("li", class_="entry inproceedings"):
        # 제목 찾기
        title_tag = entry.find("span", class_="title")
        title_text = title_tag.text.strip() if title_tag else "Unknown"

        # 저자 찾기
        authors_tags = entry.find_all("span", itemprop="author")
        authors_list = [author.find("span", itemprop="name").text.strip() for author in authors_tags]
        authors_cleaned = ", ".join(authors_list)

        # DOI 링크 찾기 (electronic edition via DOI)
        doi_tag = entry.find("a", href=re.compile(r"doi\.org"))
        if doi_tag and doi_tag.get("href"):
            pdf_link = doi_tag["href"]
        else:
            pdf_link = None

        # 논문 정보 추가
        papers.append({
            "title": title_text,
            "authors": authors_cleaned,
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers

In [95]:
url = 'https://dblp.org/db/conf/eccv/eccv2018-16.html'
DB_PATH = "con_db/ECCV_conference_2018.db"
conference_name = 'ECCV 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [96]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ECCV_2018_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [97]:
df_papers = get_www_papers('html/ECCV_2018_accepted_papers.html', conference_name)

In [98]:
df_papers.head(3)

""


In [94]:
save_to_database(df_papers, conference_name, DB_PATH)

50개의 논문이 ECCV 2018에 저장되었습니다.


# 2016

In [141]:
url = 'https://dblp.org/db/conf/eccv/eccv2016-9.html'
DB_PATH = "con_db/ECCV_conference_2016.db"
conference_name = 'ECCV 2016'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [142]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ECCV_2016_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [143]:
df_papers = get_www_papers('html/ECCV_2016_accepted_papers.html', conference_name)

In [144]:
df_papers.head()

""


In [145]:
save_to_database(df_papers, conference_name, DB_PATH)

0개의 논문이 ECCV 2016에 저장되었습니다.


# 2014

In [181]:
url = 'https://dblp.org/db/conf/eccv/eccv2014-8.html'
DB_PATH = "con_db/ECCV_conference_2014.db"
conference_name = 'ECCV 2014'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [182]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/ECCV_2014_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [183]:
df_papers = get_www_papers('html/ECCV_2014_accepted_papers.html', conference_name)

In [184]:
df_papers.head()

""


In [185]:
save_to_database(df_papers, conference_name, DB_PATH)

0개의 논문이 ECCV 2014에 저장되었습니다.
